In [1]:
import os, sys
import yaml
import toml

# Add the project root directory to sys.path
project_root = os.path.abspath("..")  # Adjust this path if needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from utils import create_file_names_list, create_file_names_list_from_csv, get_completness_status, get_blocks


2024-10-07 13:56:33.769435: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-07 13:56:34.410836: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-07 13:56:34.564230: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-07 13:56:35.688257: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-07 13:56:38.451364: W tensorflow/compiler/tf2

[10/07/24 13:56:46] WARNING  Your inference package version 0.18.1 is out of date! Please upgrade to ]8;id=404320;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=95597;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#41\41]8;;\
                             version 0.22.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

# Load datasets

In [2]:
id_tag = '0005'

# Load the configuration file
config_path = f'/workspace/home01/pipeline_configs/config_{id_tag}.toml'
config = toml.load(config_path)

# Load YAML file into a dictionary
dataset_name = config["dataset"]["name"]
dataset_yaml_file = config["dataset"]["yaml_file"]
with open(dataset_yaml_file, "r") as file:
    datasets = yaml.safe_load(file)

# get dataset
dataset = datasets[dataset_name]

# Confounder Detection

In [3]:
gt_csv_path = f"/workspace/home/data/{dataset_name}/gt_label.csv"
if "use_existing" in config["confounder_classification"]:
    confounder_pred_id = config["confounder_classification"]["use_existing"]
    print(f"Using existing confounder prediction from {confounder_pred_id}")
    if confounder_pred_id == "gt":
        pred_csv_path = dataset["gt_label_path"]
    else:
        pred_csv_path = f"/workspace/home/pipeline_outputs/{config['dataset']['name']}/{confounder_pred_id}/confounder_pred.csv"
else:
    pred_csv_path = f"/workspace/home/pipeline_outputs/{config['dataset']['name']}/{id_tag}/confounder_pred.csv"
gt_files = create_file_names_list_from_csv(gt_csv_path)


pred_files = create_file_names_list_from_csv(pred_csv_path)


blocks = get_blocks(pred_files, gt_files, 10)


Using existing confounder prediction from gt


All files processed.


In [4]:
pred_files = create_file_names_list_from_csv(pred_csv_path)

get_completness_status(pred_files, gt_files, blocks)

Number of files to be processed:  25050
Copied files:  []
Missing files:  []
Number of missing files:  0
All files processed.


True

# Segmentation

In [5]:
if "use_existing" in config["confounder_classification"]:
    confounder_pred_id = config["confounder_classification"]["use_existing"]
    print(f"Using existing confounder prediction from {confounder_pred_id}")
    if confounder_pred_id == "gt":
        cc_pred_csv_path = dataset["gt_label_path"]
    else:
        cc_pred_csv_path = f"/workspace/home/pipeline_outputs/{config['dataset']['name']}/{confounder_pred_id}/confounder_pred.csv"
else:
    cc_pred_csv_path = f"/workspace/home/pipeline_outputs/{config['dataset']['name']}/{id_tag}/confounder_pred.csv"

if "use_existing" in config["segmentation"]:
    segmentation_id = config["segmentation"]["use_existing"]
    print(f"Using existing segmentation from {segmentation_id}")
    if segmentation_id == "gt":
        segmentation_path = dataset["gt_seg_path"]
    else:
        segmentation_path = f"/workspace/home/pipeline_outputs/{dataset_name}/{segmentation_id}/seg"
else:
    segmentation_path = (
        f"/workspace/home/pipeline_outputs/{dataset_name}/{id_tag}/seg"
    )

if not os.path.exists(segmentation_path):
    seg_files = []
else:
    seg_files = create_file_names_list(segmentation_path)
    
cc_pred_files = create_file_names_list_from_csv(cc_pred_csv_path, filter_func=lambda row: row[1] == "1")

blocks = get_blocks(seg_files, cc_pred_files, 10)

Using existing confounder prediction from gt
Using existing segmentation from 0004
All files processed.


In [6]:
seg_files = create_file_names_list(segmentation_path)

get_completness_status(seg_files, cc_pred_files, blocks)

Number of files to be processed:  9210
Copied files:  []
Missing files:  []
Number of missing files:  0
All files processed.


True

# Inpainting


In [8]:
inpainted_path = f"/workspace/home/pipeline_outputs/{dataset_name}/{id_tag}/inpainting"
if "use_existing" in config["segmentation"]:
    segmentation_id = config["segmentation"]["use_existing"]
    print(f"Using existing segmentation from {segmentation_id}")
    if segmentation_id == "gt":
        segmentation_path = dataset["gt_seg_path"]
    else:
        segmentation_path = f"/workspace/home/pipeline_outputs/{dataset_name}/{segmentation_id}/seg"
else:
    segmentation_path = (
        f"/workspace/home/pipeline_outputs/{dataset_name}/{id_tag}/seg"
    )

#segmentation_path = f"/workspace/datasets/ISIC19/segmentation"

if not os.path.exists(inpainted_path):
    os.makedirs(inpainted_path)

inpainted_files = create_file_names_list(inpainted_path)
seg_files = create_file_names_list(segmentation_path)

last_range = None
blocks = get_blocks(inpainted_files, seg_files, 10)

Using existing segmentation from 0004


In [9]:
inpainted_files = create_file_names_list(inpainted_path)

last_range = get_completness_status(inpainted_files, seg_files, blocks, last_range)


Number of files to be processed:  9248
Copied files:  ['ISIC_0000781 - Copy.jpg', 'ISIC_0000783 - Copy.jpg', 'ISIC_0000785 - Copy.jpg', 'ISIC_0000788 - Copy.jpg', 'ISIC_0000941 - Copy.jpg', 'ISIC_0001004 - Copy.jpg', 'ISIC_0001174 - Copy.jpg', 'ISIC_0001456 - Copy.jpg', 'ISIC_0001474 - Copy.jpg', 'ISIC_0001756 - Copy.jpg', 'ISIC_0001799 - Copy.jpg', 'ISIC_0002157 - Copy.jpg', 'ISIC_0002201 - Copy.jpg', 'ISIC_0002550 - Copy.jpg', 'ISIC_0002819 - Copy.jpg', 'ISIC_0003199 - Copy.jpg', 'ISIC_0003343 - Copy.jpg', 'ISIC_0003348 - Copy.jpg', 'ISIC_0003405 - Copy.jpg', 'ISIC_0003559 - Copy.jpg', 'ISIC_0003622 - Copy.jpg', 'ISIC_0003715 - Copy.jpg', 'ISIC_0004768 - Copy.jpg', 'ISIC_0004821 - Copy.jpg', 'ISIC_0004880 - Copy.jpg', 'ISIC_0004934 - Copy.jpg', 'ISIC_0004937 - Copy.jpg', 'ISIC_0004950 - Copy.jpg', 'ISIC_0006005 - Copy.jpg', 'ISIC_0006023 - Copy.jpg', 'ISIC_0006783 - Copy.jpg', 'ISIC_0007396 - Copy.jpg', 'ISIC_0007504 - Copy.jpg', 'ISIC_0007522 - Copy.jpg', 'ISIC_0007993 - Copy.jpg', 